# Kaggriculture Playground

This notebook is a safe place to learn how a Kaggriculture agent works before submitting anything to Kaggle. We will run the game on this computer, make small changes, and see what happens.

Each section below explains one step. Add a Python code cell immediately below it as you work.

In [57]:
print("Hello world!!!")

Hello world!!!


## 1. Import the game simulator

The `kaggle_environments` package contains a local copy of the Kaggriculture game engine. Import the function that creates a game.

Add a code cell below this one to import the simulator.

In [58]:
# kaggle_environments is the package that contains the local game simulator.
# "make" creates one game environment for us to run.
from kaggle_environments import make

## 2. Choose the two players

A match always has two agents. Start with Kaggle's built-in `random` agent for both players. Later, replace the first player with your own `main.py` file.

Add a code cell below this one to store the agent and opponent choices in variables.

In [59]:
agent = "random"
opponent = "random"

## 3. Create a game

Create a Kaggriculture environment and choose how many turns it should last. There are 24 turns in one in-game day and 720 turns in a complete 30-day season.

Begin with 24 turns so that each experiment runs quickly. Add a code cell below this one to create the game.

In [60]:
steps = 24

env = make(
    "kaggriculture",
    configuration={"episodeSteps": steps},
    debug=True,
)

## 4. Run the match

Tell the environment to run the two players. On every turn, the game sends each player an observation, receives an action dictionary back, applies both actions, and advances the simulation.

Add a code cell below this one to run the match.

In [61]:
# perf_counter is a stopwatch designed for timing short pieces of code.
from time import perf_counter

start_time = perf_counter()
run_steps = env.run([agent, opponent])
elapsed_seconds = perf_counter() - start_time

print(f"Match completed: {len(run_steps)} turns in {elapsed_seconds:.2f} seconds")

final_step = run_steps[-1]
for player, state in enumerate(final_step):
    print(
        f"Player {player}: "
        f"reward={state.reward}, status={state.status}"
    )

Match completed: 24 turns in 0.10 seconds
Player 0: reward=2870.0, status=DONE
Player 1: reward=2930.0, status=DONE


## 5. Read the final result

After the match, the environment keeps a history of every turn. The final entry contains each player's final reward and status. A normal match should finish with a `DONE` status.

Add a code cell below this one to print the final result for both players.

In [62]:
for player, state in enumerate(env.steps[-1]):
    print(f"Player {player}: reward={state.reward}, status={state.status}")

Player 0: reward=2870.0, status=DONE
Player 1: reward=2930.0, status=DONE


## 6. Read a short match timeline

The game records every turn, but most turns are just walking around. The next cell follows player 0: it prints their farmer's grid position each turn, the action they requested, and any money change. Positions use `(x, y)`: x increases to the east (right) and y increases to the south (down).

Run this cell after the match has finished.

In [63]:
# We are following player 0 only.
player = 0
previous_money = None

# Look through the saved game one turn at a time.
for turn, turn_states in enumerate(env.steps):
    state = turn_states[player]
    observation = state.observation
    farm = observation["farms"][player]
    x, y = farm["farmer"]
    money = farm["money"]

    # This is the action player 0 requested for this turn.
    action = state.action or {}
    farmer_action = action.get("farmer", ["PASS"])
    action_text = " ".join(map(str, farmer_action))

    money_text = ""
    if previous_money is not None and money != previous_money:
        money_text = f" | money: ${previous_money:.0f} → ${money:.0f}"

    print(
        f"Turn {turn:>3} | day {observation['day'] + 1:>2}, hour {observation['hour'] + 1:>2} "
        f"| position ({x}, {y}) | action: {action_text}{money_text}"
    )
    previous_money = money

Turn   0 | day  1, hour  1 | position (4, 4) | action: PASS
Turn   1 | day  1, hour  2 | position (4, 4) | action: HARVEST
Turn   2 | day  1, hour  3 | position (5, 4) | action: EAST
Turn   3 | day  1, hour  4 | position (5, 4) | action: PASS
Turn   4 | day  1, hour  5 | position (5, 3) | action: NORTH
Turn   5 | day  1, hour  6 | position (5, 3) | action: HARVEST
Turn   6 | day  1, hour  7 | position (5, 2) | action: NORTH | money: $3000 → $2900
Turn   7 | day  1, hour  8 | position (5, 2) | action: PLANT STRAWBERRY
Turn   8 | day  1, hour  9 | position (5, 2) | action: PLANT STRAWBERRY
Turn   9 | day  1, hour 10 | position (5, 2) | action: PLANT STRAWBERRY | money: $2900 → $2890
Turn  10 | day  1, hour 11 | position (5, 2) | action: WATER
Turn  11 | day  1, hour 12 | position (5, 2) | action: PLANT STRAWBERRY
Turn  12 | day  1, hour 13 | position (5, 1) | action: NORTH
Turn  13 | day  1, hour 14 | position (5, 1) | action: PLANT WHEAT | money: $2890 → $2870
Turn  14 | day  1, hour 15

In [42]:
print("hello!")

hello!


## 7. Try your own agent

Once you create `main.py`, change the first player from `random` to `main.py`. Run a short game first, then use 720 turns for a full season.

Keep the built-in `random` opponent at first. Later, compare your agent with `starter` and with older versions of your own agent.

## 8. What to learn from each run

Do not worry about winning immediately. First check that your agent finishes every game, returns valid actions, and does not let crops die or inventory overflow. Then make one small change at a time and compare the results across several games.